# 🤖 Customer Churn Prediction – Model Training & Evaluation

> **Models**: Random Forest · XGBoost  
> **Target metrics**: Accuracy ≥ 86% · AUC ≥ 0.92

---

## Table of Contents
1. [Setup](#1)
2. [Load & Split Data](#2)
3. [Train Random Forest](#3)
4. [Train XGBoost](#4)
5. [ROC & PR Curves](#5)
6. [Confusion Matrices](#6)
7. [Feature Importance](#7)
8. [Model Comparison](#8)
9. [Threshold Analysis](#9)
10. [Score New Customers (Demo)](#10)

---
## 1. Setup <a id='1'></a>

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split

from src.data_generation import generate_dataset
from src.feature_engineering import add_domain_features, split_X_y, get_feature_names
from src.train import train_random_forest, train_xgboost
from src.evaluate import evaluate_model, compare_models, threshold_search, shap_summary
from src.visualize import (
    plot_roc_curves, plot_precision_recall_curves,
    plot_confusion_matrix, plot_feature_importance,
    plot_model_comparison, plot_threshold_analysis,
)

FIG_DIR = '../reports/figures/models'
RANDOM_STATE = 42
THRESHOLD = 0.40

print('✅ Setup complete!')

---
## 2. Load & Split Data <a id='2'></a>

In [ ]:
df = generate_dataset(n_samples=10_000, random_state=RANDOM_STATE)
X, y = split_X_y(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.125, random_state=RANDOM_STATE, stratify=y_train
)

print(f'Train : {len(X_train):,} rows  |  churn rate: {y_train.mean():.2%}')
print(f'Val   : {len(X_val):,}  rows  |  churn rate: {y_val.mean():.2%}')
print(f'Test  : {len(X_test):,} rows  |  churn rate: {y_test.mean():.2%}')

---
## 3. Train Random Forest <a id='3'></a>

In [ ]:
rf_pipe = train_random_forest(
    X_train, y_train,
    params={'n_estimators': 300, 'max_depth': 12, 'random_state': RANDOM_STATE},
    run_cv=True,
    save_path='../models/random_forest_pipeline.pkl',
)
print('Random Forest trained ✅')

---
## 4. Train XGBoost <a id='4'></a>

In [ ]:
xgb_pipe = train_xgboost(
    X_train, y_train,
    params={'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05,
            'scale_pos_weight': 2.8, 'random_state': RANDOM_STATE},
    run_cv=True,
    save_path='../models/xgboost_pipeline.pkl',
)
print('XGBoost trained ✅')

---
## 5. ROC & PR Curves <a id='5'></a>

In [ ]:
pipelines = {'Random Forest': rf_pipe, 'XGBoost': xgb_pipe}

fig = plot_roc_curves(pipelines, X_test, y_test,
                      save_path=f'{FIG_DIR}/roc_curves.png')
plt.show()

In [ ]:
fig = plot_precision_recall_curves(pipelines, X_test, y_test,
                                    save_path=f'{FIG_DIR}/pr_curves.png')
plt.show()

---
## 6. Confusion Matrices <a id='6'></a>

In [ ]:
# Optimise threshold on validation set
opt_t = threshold_search(xgb_pipe, X_val, y_val, metric='f1')
print(f'Optimal threshold: {opt_t:.3f}')

fig = plot_confusion_matrix(rf_pipe, X_test, y_test, opt_t,
                             'Random Forest',
                             save_path=f'{FIG_DIR}/rf_confusion_matrix.png')
plt.show()

fig = plot_confusion_matrix(xgb_pipe, X_test, y_test, opt_t,
                             'XGBoost',
                             save_path=f'{FIG_DIR}/xgb_confusion_matrix.png')
plt.show()

---
## 7. Feature Importance <a id='7'></a>

In [ ]:
X_eng = add_domain_features(X_test)
preprocessor = rf_pipe.named_steps['preprocessor']
feature_names = list(preprocessor.get_feature_names_out())

fig = plot_feature_importance(rf_pipe, feature_names, top_n=20,
                               model_name='Random Forest',
                               save_path=f'{FIG_DIR}/rf_feature_importance.png')
plt.show()

fig = plot_feature_importance(xgb_pipe, feature_names, top_n=20,
                               model_name='XGBoost',
                               save_path=f'{FIG_DIR}/xgb_feature_importance.png')
plt.show()

---
## 8. Model Comparison <a id='8'></a>

In [ ]:
cmp_df = compare_models(pipelines, X_test, y_test, threshold=opt_t)
display(cmp_df[['model','accuracy','roc_auc','pr_auc','f1','precision','recall']].round(4))

fig = plot_model_comparison(cmp_df, save_path=f'{FIG_DIR}/model_comparison.png')
plt.show()

winner = cmp_df.iloc[0]['model']
print(f'\n🏆 Best model: {winner}  |  AUC={cmp_df.iloc[0]["roc_auc"]:.4f}')

---
## 9. Threshold Analysis <a id='9'></a>

In [ ]:
fig = plot_threshold_analysis(xgb_pipe, X_val, y_val,
                               model_name='XGBoost',
                               save_path=f'{FIG_DIR}/threshold_analysis.png')
plt.show()

---
## 10. Score New Customers (Demo) <a id='10'></a>

In [ ]:
from src.predict import score_customers

# Use 10 test records as "new" customers
new_customers = X_test.head(10).copy()
new_customers['customer_id'] = [f'NEW-{i:04d}' for i in range(10)]

results = score_customers(xgb_pipe, new_customers, threshold=opt_t)
display(results)

print('\n✅ Inference complete. High-risk customers flagged for retention action.')